In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1/generation_config.json


# ClimateCalendar — Day 2: Function-Calling Agent

Demonstrates Gemma 4 chaining multiple tool calls (ENSO + climate trend +
projection + soil) to answer a complex agronomic query.

The model decides which tools to call. Python executes them. Gemma writes
the final answer grounded in real fetched data.

**Setup:** After running Cell 2, factory-reset the kernel before continuing.

In [2]:
!pip install -q accelerate safetensors tokenizers 
!pip install -q --force-reinstall --no-deps git+https://github.com/huggingface/transformers.git
!pip install -q "huggingface_hub>=1.5.0,<2.0" accelerate safetensors tokenizers librosa torchvision bitsandbytes
print("✓ Install complete. NOW: Run menu → Factory Reset Session.")
print("✓ Install complete. NOW: Run menu → Factory Reset Session.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 16.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 85.1 MB/s eta 0:00:00:00:01
✓ Install complete. NOW: Run menu → Factory Reset Session.
✓ Install complete. NOW: Run menu → Factory Reset Session.


In [18]:
# Wipe any stale clone, pull fresh from GitHub, clear Python's import cache
import shutil, os, subprocess, sys

REPO_PATH = "/kaggle/working/climate-calendar"
shutil.rmtree(REPO_PATH, ignore_errors=True)

result = subprocess.run(
    ["git", "clone", "https://github.com/tkaushik015/climate-calendar.git", REPO_PATH],
    capture_output=True, text=True,
)
print("Clone return code:", result.returncode)
if result.returncode != 0:
    print("Error:", result.stderr)

# Verify the files we need are present
print("\nFiles in src/:")
print(sorted(os.listdir(f"{REPO_PATH}/src")))
print("\nFiles in src/tools/:")
print(sorted(os.listdir(f"{REPO_PATH}/src/tools")))

# Add repo to path
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Clear any previously-cached imports of our modules
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("src"):
        del sys.modules[mod_name]

print("\n✓ Repo cloned, sys.path updated, cache cleared")

Clone return code: 0

Files in src/:
['__init__.py', '__pycache__', 'agent.py', 'tools']

Files in src/tools/:
['__init__.py', '__pycache__', 'climate_projection.py', 'climate_trend.py', 'enso.py', 'soil_profile.py']

✓ Repo cloned, sys.path updated, cache cleared


In [19]:
import sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("src"):
        del sys.modules[mod_name]
print("✓ Cleared cached src.* modules")

# Also delete any pyc files that might exist from the old version
import shutil
shutil.rmtree("/kaggle/working/climate-calendar/src/__pycache__", ignore_errors=True)
shutil.rmtree("/kaggle/working/climate-calendar/src/tools/__pycache__", ignore_errors=True)
print("✓ Removed pycache files")

✓ Cleared cached src.* modules
✓ Removed pycache files


## Step 1: Load Gemma 4 E4B in 4-bit

Same model load as Day 1.5 — 4-bit quantization for ~4GB VRAM and fast iteration.

In [4]:
import kagglehub
import torch
import time
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e4b-it")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb,
    device_map="auto",
)

def ask_gemma(messages, max_new_tokens=250):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=model.dtype)
    input_len = inputs["input_ids"].shape[-1]
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=1.0, top_p=0.95, top_k=64,
        use_cache=True,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    return processor.decode(out[0][input_len:], skip_special_tokens=True)

print(f"✓ Gemma 4 E4B loaded ({round(torch.cuda.memory_allocated() / 1e9, 2)} GB VRAM)")
print(f"✓ Device: {next(model.parameters()).device}")

t0 = time.time()
_ = ask_gemma([{"role": "user", "content": "Say 'ready' in three words."}], max_new_tokens=15)
print(f"✓ Inference speed: ~{15 / (time.time() - t0):.1f} tokens/sec")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✓ Gemma 4 E4B loaded (0.0 GB VRAM)
✓ Device: cuda:1
✓ Inference speed: ~7.1 tokens/sec


## Step 2: Smoke-test all four tools in Kaggle

Confirms each data source is reachable from Kaggle's environment before
we wire them into Gemma's reasoning loop.

In [7]:
from src.tools.climate_trend import get_climate_trend
from src.tools.climate_projection import get_climate_projection
from src.tools.enso import get_enso_state
from src.tools.soil_profile import get_soil_profile

print("ENSO:", get_enso_state()["explanation"])
print()
print("Soil:", get_soil_profile(30.21, 74.94)["explanation"])
print()
print("Climate trend (last 3 years):")
for row in get_climate_trend(30.21, 74.94, 2022, 2024):
    print(" ", row)
print()
print("Climate projection (2050):")
proj = get_climate_projection(30.21, 74.94, 2050, 2050)
print(" ", proj[0] if proj else "no data")

ENSO: As of FEB 2026, ENSO is in a Neutral phase (ONI -0.16°C).

Soil: Topsoil at (30.21, 74.94) is loamy (well-balanced). Soil pH is in the agronomic sweet spot. Organic matter is low — prioritize compost / cover crops. (Sampled from a point ~2.4 km away.)

Climate trend (last 3 years):
  {'year': 2022, 'temperature_2m_mean': 24.68712328767123, 'precipitation_sum': 561.2}
  {'year': 2023, 'temperature_2m_mean': 24.24931506849315, 'precipitation_sum': 511.1}
  {'year': 2024, 'temperature_2m_mean': 24.49863387978142, 'precipitation_sum': 408.8}

Climate projection (2050):
  {'year': 2050, 'temperature_2m_mean': 26.85, 'precipitation_sum': 585.7}


## Step 3: The Agent Demo

A complex multi-tool query. Gemma 4 decides which tools to call, executes
them via function-calling, then synthesizes a grounded recommendation for
Ramesh Singh.

In [20]:
from src.agent import run_agent

query = (
    "I am Ramesh Singh, a wheat farmer in Bathinda, Punjab "
    "(latitude 30.21, longitude 74.94). My family has always planted on November 5. "
    "Given the current ENSO state, my historical climate (1995-2024), the projected "
    "climate for 2025-2035, and my soil profile — should I keep the November 5 date "
    "this year, or shift it? Give me a 5-7 day window and explain your reasoning. "
    "Cite the data you used."
)

answer = run_agent(query, processor, model)
print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(answer)


--- Step 1: tool selection raw response ---
call:get_enso_state{}call:get_climate_trend{end_year:2024,latitude:30.21,longitude:74.94,start_year:1995}call:get_climate_projection{end_year:2035,latitude:30.21,longitude:74.94,start_year:2025}call:get_soil_profile{latitude:30.21,longitude:74.94}
--- end raw ---

  → tool call: get_enso_state({})
  ← tool result: {"latest_year": 2026, "latest_month": "FEB", "latest_oni": -0.16, "classification": "Neutral", "intensity": "\u2014", "explanation": "As of FEB 2026, ENSO is in a Neutral phase (ONI -0.16\u00b0C)."}
  → tool call: get_climate_trend({'end_year': 2024, 'latitude': 30.21, 'longitude': 74.94, 'start_year': 1995})
  ← tool result: [{"year": 1995, "temperature_2m_mean": 24.380273972602737, "precipitation_sum": 766.0}, {"year": 1996, "temperature_2m_mean": 24.232513661202187, "precipitation_sum": 651.8}, {"year": 1997, "temperatu...
  → tool call: get_climate_projection({'end_year': 2035, 'latitude': 30.21, 'longitude': 74.94, 'start_year